# Raw data and cohort definition


It writes metadata/samples.csv, which the papermill driver reads to select
the samples passed to notebook 001.

## Setup

In [2]:
import os
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 80)

In [3]:
def find_project(start=None):
    here = Path(start or Path.cwd()).resolve()
    for parent in [here, *here.parents]:
        if (parent / "Data").is_dir() and (parent / "metadata").is_dir():
            return parent
    raise FileNotFoundError("Set CAP_PROJECT to the project data directory")


def find_publication():
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        for candidate in (parent, parent / "publication_notebooks"):
            if (candidate / "scripts" / "publication_figure_state.py").is_file():
                if (candidate / "data" / "cancer_meta_programme_signatures.xlsx").is_file():
                    return candidate
    raise FileNotFoundError("Run from the publication_notebooks directory")


In [ ]:
# the GEO download sits outside the project folder, so its location is an environment variable rather than a hard-coded path
project = Path(os.environ.get("CAP_PROJECT") or find_project()).expanduser().resolve()
raw_dir = Path(os.environ.get("CAP_RAW_DATA", "~/Downloads/Project_Nowotny/Helena/raw_counts")).expanduser().resolve()

features_file = raw_dir.parent / "features.tsv"
sample_sheet = project / "metadata" / "samples.csv"

In [5]:
assert raw_dir.is_dir(), f"raw counts not found at {raw_dir}; set CAP_RAW_DATA"
assert features_file.is_file(), f"features.tsv not found at {features_file}"

In [6]:
def display_path(path):
    """project-relative where possible, so an external absolute path is never printed"""
    path = Path(path).resolve()
    try:
        return path.relative_to(project)
    except ValueError:
        return Path("<external raw data>") / path.name


def sample_paths(sample_id):
    return (raw_dir / f"{sample_id}-matrix.mtx.gz",
            raw_dir / f"{sample_id}-barcodes.tsv.gz")

In [7]:
matrices = sorted(raw_dir.glob("*-matrix.mtx.gz"))
barcodes = sorted(raw_dir.glob("*-barcodes.tsv.gz"))

assert matrices, f"no matrix files under {display_path(raw_dir)}"
assert len(matrices) == len(barcodes), (len(matrices), len(barcodes))

print(f"raw counts directory : {display_path(raw_dir)}")
print(f"matrix files         : {len(matrices)}")
print(f"barcode files        : {len(barcodes)}")
print(f"shared features.tsv  : {display_path(features_file)}")

raw counts directory : <external raw data>/raw_counts
matrix files         : 69
barcode files        : 69
shared features.tsv  : <external raw data>/features.tsv


In [8]:
accessions = sorted({p.name.split("_")[0][:7] for p in matrices})
print("accession prefixes present:", accessions)

example = matrices[0].name
print(f"\nexample filename: {example}")

accession prefixes present: ['GSM4909']

example filename: GSM4909253_N-PM0092-Total-matrix.mtx.gz


In [9]:
def parse_sample_name(stem):
    """Turn 'GSM4909285_TN-B1-MH4031' into structured metadata."""
    conditions = {"N", "TN", "HER2", "ER", "mER", "B1"}
    sort_labels = {"Total", "Epi"}
    site_labels = {"LN"}

    gsm, _, rest = stem.partition("_")
    tokens = rest.split("-")

    condition = tokens[0]
    if condition not in conditions:
        raise ValueError(f"unrecognised condition {condition!r} in {stem!r}")

    # 'TN-B1-MH4031' is a BRCA1-mutant tumour; 'B1-KCF0894' is preneoplastic tissue, where B1 is itself the condition.
    idx, brca1 = 1, condition == "B1"
    if len(tokens) > 1 and tokens[1] == "B1":
        brca1, idx = True, 2

    patient = tokens[idx]
    qualifiers = tokens[idx + 1:]

    return {
        "sample_id": stem,
        "gsm": gsm,
        "condition": condition,
        "patient": patient,
        "brca1_carrier": brca1,
        "sorting": next((q for q in qualifiers if q in sort_labels), "unsorted"),
        "site": "lymph_node" if any(q in site_labels for q in qualifiers) else "primary",
        "replicate": next((q for q in qualifiers
                           if q not in sort_labels and q not in site_labels), ""),
    }


# one worked example, so the parsing is visible rather than implied
demo = "GSM4909285_TN-B1-MH4031"
for key, value in parse_sample_name(demo).items():
    print(f"  {key:16s} {value}")

  sample_id        GSM4909285_TN-B1-MH4031
  gsm              GSM4909285
  condition        TN
  patient          MH4031
  brca1_carrier    True
  sorting          unsorted
  site             primary
  replicate        


In [10]:
samples = pd.DataFrame(
    [parse_sample_name(p.name.replace("-matrix.mtx.gz", "")) for p in matrices]
)

print(f"{len(samples)} samples\n")
print(pd.crosstab(samples["condition"], samples["sorting"]).to_string())
print()
print(pd.crosstab(samples["condition"], samples["site"]).to_string())

69 samples

sorting    Epi  Total  unsorted
condition                      
B1           0      0         4
ER           0      0        24
HER2         0      0         6
N           11     13         0
TN           0      0         8
mER          0      0         3

site       lymph_node  primary
condition                     
B1                  0        4
ER                  6       18
HER2                0        6
N                   0       24
TN                  0        8
mER                 1        2


In [11]:
def decide_inclusion(row, has_total):
    """rules are applied in order, so a sample is counted under the first that matches"""
    if row["condition"] == "mER":
        return False, "male breast cancer; distinct biology and sex chromosome complement"

    if row["condition"] == "B1":
        return False, "BRCA1 preneoplastic tissue; neither normal nor tumour"

    if row["site"] == "lymph_node":
        return False, "lymph node metastasis; cohort restricted to primary tissue"

    if row["sorting"] == "Epi":
        if row["patient"] in has_total:
            return False, "epithelially sorted; unsorted -Total sample used for this patient"
        return False, ("epithelially sorted; cell-type composition not comparable "
                       "with the unsorted -Total samples in the rest of the cohort")

    return True, ""

In [12]:
has_total = set(samples.loc[samples["sorting"] == "Total", "patient"])

samples["include"], samples["exclusion_reason"] = zip(
    *samples.apply(lambda row: decide_inclusion(row, has_total), axis=1)
)

included = samples[samples["include"]]

print(f"{len(samples)} samples found -> {len(included)} included\n")
print("included:")
print(included.groupby("condition").agg(
    n_samples=("sample_id", "size"),
    n_patients=("patient", "nunique"),
    n_brca1=("brca1_carrier", "sum"),
).to_string())

print("\nexcluded:")
for reason, n in samples.loc[~samples["include"], "exclusion_reason"].value_counts().items():
    print(f"  {n:3d}  {reason}")

69 samples found -> 45 included

included:
           n_samples  n_patients  n_brca1
condition                                
ER                18          17        0
HER2               6           6        0
N                 13          13        0
TN                 8           8        4

excluded:
    6  epithelially sorted; unsorted -Total sample used for this patient
    6  lymph node metastasis; cohort restricted to primary tissue
    5  epithelially sorted; cell-type composition not comparable with the unsorted -Total samples in the rest of the cohort
    4  BRCA1 preneoplastic tissue; neither normal nor tumour
    3  male breast cancer; distinct biology and sex chromosome complement


In [ ]:
repeated = (included[included.duplicated("patient", keep=False)]
            .sort_values(["patient", "sample_id"]))

print(f"{len(included)} samples from {included['patient'].nunique()} distinct patients\n")

for patient, group in repeated.groupby("patient"):
    conditions = sorted(set(group["condition"]))
    flag = "    -> spans conditions" if len(conditions) > 1 else ""
    print(f"  {patient:10s} {list(group['sample_id'])}{flag}")

45 samples from 42 distinct patients

  MH0029     ['GSM4909305_ER-MH0029-7C', 'GSM4909306_ER-MH0029-9C']
  MH0064     ['GSM4909263_N-MH0064-Total', 'GSM4909313_ER-MH0064-T']   <-- SPANS CONDITIONS
  MH0114     ['GSM4909284_TN-MH0114-T2', 'GSM4909299_ER-MH0114-T3']   <-- SPANS CONDITIONS


##  Write the sample sheet

In [14]:
# both files must exist for everything we intend to load
def files_present(sample_id):
    matrix, barcode = sample_paths(sample_id)
    return matrix.exists() and barcode.exists()

samples["files_present"] = samples["sample_id"].map(files_present)
missing = samples.loc[samples["include"] & ~samples["files_present"], "sample_id"]
assert missing.empty, f"missing raw files for: {list(missing)}"

samples = samples.sort_values(["condition", "patient", "sample_id"]).reset_index(drop=True)
sample_sheet.parent.mkdir(parents=True, exist_ok=True)
samples.to_csv(sample_sheet, index=False)

# the cohort reported in the methods; these fail loudly if the rules or the download ever change
assert len(samples) == 69, len(samples)
assert samples["include"].sum() == 45, samples["include"].sum()
assert included["patient"].nunique() == 42, included["patient"].nunique()

print(f"written: {display_path(sample_sheet)}")
print(f"  {len(samples)} rows, {samples['include'].sum()} included")
print(f"  columns: {', '.join(samples.columns)}")

written: metadata/samples.csv
  69 rows, 45 included
  columns: sample_id, gsm, condition, patient, brca1_carrier, sorting, site, replicate, include, exclusion_reason, files_present


In [15]:
size_mb = {
    sample_id: sample_paths(sample_id)[0].stat().st_size / 1e6
    for sample_id in included["sample_id"]
}

sizes = (included[["sample_id", "condition"]]
         .assign(size_mb=lambda d: d["sample_id"].map(size_mb))
         .sort_values("size_mb"))

print("smallest samples, good candidates for demo_id:\n")
print(sizes.head(5).to_string(index=False, float_format=lambda v: f"{v:.1f}"))

print(f"\nsuggested:\n    demo_id = \"{sizes.iloc[0]['sample_id']}\"")

smallest samples, good candidates for demo_id:

                sample_id condition  size_mb
   GSM4909292_HER2-MH0069      HER2      2.4
   GSM4909311_ER-MH0056-T        ER      6.4
     GSM4909283_TN-SH0106        TN      6.7
     GSM4909300_ER-MH0032        ER      7.0
GSM4909272_N-MH0021-Total         N      7.2

suggested:
    demo_id = "GSM4909292_HER2-MH0069"


In [ ]:
import session_info


In [ ]:

session_info.show()